# ATC / ADAPT — SFT + GRPO (medium tier): Colab or Jupyter

Same training recipe as the Colab notebook (`atc_colab_sft_grpo_medium.ipynb`): optional SFT cold-start, then GRPO with **ADAPT_FOCUS** and the same hyperparameter blocks.

**Runtime:** This file auto-detects **Google Colab** vs **local Jupyter**. On Jupyter it defaults to the **current checkout** (no fresh clone) and writes outputs under a local directory instead of Drive.

**Friend's Colab reference:** keep using the original Colab-only notebook on Google; use this copy when running on your own Jupyter server.


## 0 — Hugging Face token (optional)

**Colab:** Sidebar → **Secrets** → `HF_TOKEN`.

**Jupyter:** Export `HF_TOKEN` or `HUGGING_FACE_HUB_TOKEN` in the shell before starting Jupyter, or paste into the environment in your IDE.


In [ ]:
try:
    from google.colab import userdata
    IS_COLAB = True
except Exception:
    IS_COLAB = False

import os
if IS_COLAB:
    try:
        t = userdata.get("HF_TOKEN")
        if t:
            os.environ["HF_TOKEN"] = t
            os.environ["HUGGING_FACE_HUB_TOKEN"] = t
            print("HF_TOKEN loaded from Colab secrets.")
        else:
            print("No HF_TOKEN secret — add one for gated models / router.")
    except Exception as e:
        print("Colab secrets error:", e)
else:
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        print("HF token present in environment.")
    else:
        print("No HF_TOKEN in env — set if you need gated Hub models.")


## 1 — Paths and hyperparameters

**Jupyter defaults:** `USE_LOCAL_REPO=True` uses this repo on disk (no `git clone`). Set `USE_LOCAL_REPO=False` to clone `REPO_URL` into `REPO_DIR` like Colab.

**Outputs:** On Jupyter, `OUTPUT_DIR` defaults to `~/atc-grpo-medium` unless you override below.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import userdata  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# --- Git (only used when USE_LOCAL_REPO is False) ---------------------------
REPO_URL = "https://github.com/yvishi/ats.git"
GIT_BRANCH = "cleaned"

# Colab layout matches friend's notebook; Jupyter discovers repo root.
if IS_COLAB:
    REPO_DIR = Path("/content/ATS")
    USE_LOCAL_REPO = False
else:
    # On a bare Jupyter host, set USE_LOCAL_REPO = False to clone REPO_URL into REPO_DIR.
    USE_LOCAL_REPO = True
    _cwd = Path.cwd().resolve()
    if (_cwd / "training" / "train_grpo.py").exists():
        REPO_DIR = _cwd
    elif (_cwd.parent / "training" / "train_grpo.py").exists():
        REPO_DIR = _cwd.parent
    else:
        REPO_DIR = _cwd
        print("WARN: training/train_grpo.py not found relative to cwd; set REPO_DIR manually.")

USE_DRIVE = IS_COLAB and True

if IS_COLAB:
    OUTPUT_DIR = Path("/content/drive/MyDrive/atc-grpo-medium") if USE_DRIVE else Path("/content/atc-grpo-medium")
    SFT_DATA = Path("/content/drive/MyDrive/atc-sft-data.jsonl") if USE_DRIVE else Path("/content/atc-sft-data.jsonl")
    SFT_ADAPTER_DIR = Path("/content/drive/MyDrive/atc-sft-json-adapter") if USE_DRIVE else Path("/content/atc-sft-json-adapter")
else:
    _base = Path.home() / "atc-grpo-medium"
    OUTPUT_DIR = _base
    SFT_DATA = _base.parent / "atc-sft-data.jsonl"
    SFT_ADAPTER_DIR = _base.parent / "atc-sft-json-adapter"

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
SEED = 42
LORA_RANK = 16

QUICK_ANALYSIS = True
RUN_SFT = True

ADAPT_FOCUS = True
DOMAIN_EPISODE_RATIO = 0.55
STAGE_EPOCH_SCALE = 0.2
ADAPT_EVAL_EPISODES = 4

if QUICK_ANALYSIS:
    SFT_EPISODES = 200
    SFT_EPOCHS = 0.5
    SFT_MAX_SEQ_LENGTH = 1024
    SFT_BATCH_SIZE = 1
    SFT_GRAD_ACCUM = 4

    GRPO_EPISODES = 40
    GRPO_BATCH_SIZE = 2
    GRPO_GRAD_ACCUM = 1
    GRPO_N_GENERATIONS = 4
    GRPO_MAX_NEW_TOKENS = 256
    GRPO_LOGGING_STEPS = 1

    EVAL_EPISODES = 3
    RUN_EVAL = True
else:
    SFT_EPISODES = 800
    SFT_EPOCHS = 1.0
    SFT_MAX_SEQ_LENGTH = 2048
    SFT_BATCH_SIZE = 1
    SFT_GRAD_ACCUM = 8

    GRPO_EPISODES = 150
    GRPO_BATCH_SIZE = 4
    GRPO_GRAD_ACCUM = 2
    GRPO_N_GENERATIONS = 8
    GRPO_MAX_NEW_TOKENS = 384
    GRPO_LOGGING_STEPS = 1

    EVAL_EPISODES = 8
    RUN_EVAL = True

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

print("IS_COLAB", IS_COLAB, "USE_LOCAL_REPO", USE_LOCAL_REPO, "REPO_DIR", REPO_DIR)
print("OUTPUT_DIR", OUTPUT_DIR)
print("QUICK_ANALYSIS", QUICK_ANALYSIS, "ADAPT_FOCUS", ADAPT_FOCUS)
print("RUN_SFT", RUN_SFT, "SFT_EPOCHS", SFT_EPOCHS, "SFT_EPISODES", SFT_EPISODES)
print("GRPO_EPISODES", GRPO_EPISODES, "B", GRPO_BATCH_SIZE, "GA", GRPO_GRAD_ACCUM, "G", GRPO_N_GENERATIONS)
print("ADAPT", "ratio", DOMAIN_EPISODE_RATIO, "stage_scale", STAGE_EPOCH_SCALE, "adapt_eval", ADAPT_EVAL_EPISODES)
print("RUN_EVAL", RUN_EVAL, "EVAL_EPISODES", EVAL_EPISODES)


In [ ]:
import os
from pathlib import Path

if IS_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SFT_ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
print("Ready:", OUTPUT_DIR)


## 2 — Clone repository (skipped on local Jupyter when `USE_LOCAL_REPO=True`)


In [ ]:
import shutil, subprocess, os, sys
from pathlib import Path

if USE_LOCAL_REPO:
    if not (REPO_DIR / "training" / "train_grpo.py").exists():
        raise FileNotFoundError(f"Expected repo at {REPO_DIR}; open notebook from repo root or set REPO_DIR.")
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    print("Using local repo, cwd:", Path.cwd())
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    print("Cloned, cwd:", Path.cwd())


## 3 — Install dependencies (Unsloth + TRL)

**Colab:** May auto-restart the runtime once (same as friend's notebook).

**Jupyter:** Does not `kill` the kernel. If imports fail after this cell, **restart the kernel once** manually and continue from §2.


In [ ]:
import os, subprocess, sys
from pathlib import Path

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)

pip("pip")
pip("unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git")
pip("trl>=0.9.6", "datasets>=2.20.0", "accelerate>=0.32.0", "peft>=0.12.0", "bitsandbytes>=0.43.0")
pip("matplotlib>=3.9.0", "numpy>=1.26.0")
pip("openenv-core[core]>=0.2.3", "fastapi>=0.128.0", "openai>=2.30.0", "pydantic>=2.12.0", "uvicorn>=0.41.0")
print("pip install complete")

if IS_COLAB:
    restart_marker = Path("/content/.ats_runtime_restarted_after_install")
    if not restart_marker.exists():
        restart_marker.write_text("1", encoding="utf-8")
        print("Restarting runtime once to reload binary deps cleanly...", flush=True)
        os.kill(os.getpid(), 9)
    else:
        print("Restart already performed after install; continuing.")
else:
    marker = REPO_DIR / ".ats_runtime_restarted_after_install"
    if not marker.exists():
        marker.write_text("1", encoding="utf-8")
        print("First install done. If the next cells fail to import Unsloth, use Kernel → Restart, then re-run from §1 onward (skip this install cell after restart).")
    else:
        print("Install marker present; assuming deps already loaded in this environment.")


## 4 — GPU check + GRPO dataset sanity


In [ ]:
import torch
from training.dataset import build_episode_dataset

assert torch.cuda.is_available(), "Need a CUDA GPU for this notebook."
print("GPU:", torch.cuda.get_device_name(0))

rows = build_episode_dataset(
    n_episodes=4,
    seed=SEED,
    include_adapt=True,
    include_generator=False,
    include_supervisor=False,
)
print("rows:", len(rows), "roles:", sorted({r["agent_role"] for r in rows}))


## 5 — Phase A (optional): SFT on strict JSON


In [ ]:
import subprocess, sys

if not RUN_SFT:
    print("Skipping SFT (set RUN_SFT=True to enable).")
else:
    r0 = subprocess.run([
        sys.executable, "training/build_sft_dataset.py",
        "--out", str(SFT_DATA),
        "--episodes", str(SFT_EPISODES),
        "--seed", str(SEED),
    ], cwd=str(REPO_DIR), capture_output=True, text=True)
    if r0.returncode != 0:
        print(r0.stdout)
        print(r0.stderr)
        r0.check_returncode()
    _old_argv = sys.argv[:]
    try:
        sys.argv = [
            "train_sft.py",
            "--dataset", str(SFT_DATA),
            "--output_dir", str(SFT_ADAPTER_DIR),
            "--model", MODEL_NAME,
            "--epochs", str(SFT_EPOCHS),
            "--max_seq_length", str(SFT_MAX_SEQ_LENGTH),
            "--batch_size", str(SFT_BATCH_SIZE),
            "--grad_accum", str(SFT_GRAD_ACCUM),
            "--lora_rank", str(LORA_RANK),
            "--seed", str(SEED),
            "--trainer", "hf",
        ]
        print("Running:", " ".join([sys.executable] + sys.argv), flush=True)
        from training.train_sft import main as _train_sft_main
        _train_sft_main()
    finally:
        sys.argv = _old_argv
    print("SFT adapter:", SFT_ADAPTER_DIR)


## 6 — Phase B: GRPO multi-agent training


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("PYTHONUNBUFFERED", "1")

try:
    import unsloth  # noqa: F401
except Exception as e:
    raise RuntimeError(
        "Unsloth failed to import. On Colab: re-run install, let it restart, then run from section 2. "
        "On Jupyter: restart kernel once after first pip install, then re-run from section 1."
    ) from e

_old_argv = sys.argv[:]
try:
    sys.argv = [
        "train_grpo.py",
        "--model", MODEL_NAME,
        "--output_dir", str(OUTPUT_DIR),
        "--episodes", str(GRPO_EPISODES),
        "--lora_rank", str(LORA_RANK),
        "--seed", str(SEED),
        "--eval_episodes", str(EVAL_EPISODES),
        "--batch_size", str(GRPO_BATCH_SIZE),
        "--grad_accum", str(GRPO_GRAD_ACCUM),
        "--n_generations", str(GRPO_N_GENERATIONS),
        "--max_new_tokens", str(GRPO_MAX_NEW_TOKENS),
        "--logging_steps", str(GRPO_LOGGING_STEPS),
    ]
    if not RUN_EVAL:
        sys.argv.append("--no_eval")
    sys.argv.extend([
        "--domain_episode_ratio", str(DOMAIN_EPISODE_RATIO),
        "--stage_epoch_scale", str(STAGE_EPOCH_SCALE),
        "--adapt_eval_episodes", str(ADAPT_EVAL_EPISODES),
    ])
    if ADAPT_FOCUS:
        sys.argv.append("--adapt_focus")
    _sft_cfg = Path(SFT_ADAPTER_DIR) / "adapter_config.json"
    if _sft_cfg.exists():
        sys.argv.extend(["--sft_adapter", str(SFT_ADAPTER_DIR)])
        print("GRPO will load SFT adapter:", SFT_ADAPTER_DIR, flush=True)
    else:
        print("No SFT adapter at", _sft_cfg, "(run section 5 first or set RUN_SFT=True).", flush=True)
    print("Running:", " ".join([sys.executable] + sys.argv), flush=True)
    from training.train_grpo import main as _train_grpo_main
    _train_grpo_main()
finally:
    sys.argv = _old_argv

print("Saved:", OUTPUT_DIR)


## 7 — Evaluation: base vs trained (`training/eval.py`)


In [ ]:
import subprocess, sys
from pathlib import Path

if not RUN_EVAL:
    print("Skipping eval cell for quick analysis (set RUN_EVAL=True to enable).")
else:
    eval_json = OUTPUT_DIR / "eval_results_colab.json"
    subprocess.run([
        sys.executable, "training/eval.py",
        "--base", MODEL_NAME,
        "--trained", str(OUTPUT_DIR),
        "--episodes", str(EVAL_EPISODES),
        "--output", str(eval_json),
    ], cwd=str(REPO_DIR), check=True)
    print(eval_json.read_text()[:2000])


## 8 — Reward curves and eval plots


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "training/plot_rewards.py",
    "--input", str(OUTPUT_DIR / "reward_curves.json"),
    "--save", str(plots_dir),
    "--no_show",
], cwd=str(REPO_DIR), check=False)

_ba = OUTPUT_DIR / "grpo_before_after.json"
_eval = OUTPUT_DIR / "eval_results_colab.json" if (OUTPUT_DIR / "eval_results_colab.json").exists() else OUTPUT_DIR / "eval_results.json"
if _ba.exists():
    subprocess.run([
        sys.executable, "training/plot_rewards.py",
        "--eval_results", str(_ba),
        "--save", str(plots_dir / "before_after_grpo"),
        "--no_show",
    ], cwd=str(REPO_DIR), check=False)
elif _eval.exists():
    subprocess.run([
        sys.executable, "training/plot_rewards.py",
        "--eval_results", str(_eval),
        "--save", str(plots_dir),
        "--no_show",
    ], cwd=str(REPO_DIR), check=False)

for name in ("training_curves.png",):
    p = plots_dir / name
    if p.exists():
        display(Image(filename=str(p)))
for sub, name in ((plots_dir, "eval_comparison.png"), (plots_dir / "before_after_grpo", "eval_comparison.png")):
    p = sub / name
    if p.exists():
        display(Image(filename=str(p)))


## 9 — Optional: push adapter to Hugging Face Hub


In [ ]:
# from huggingface_hub import HfApi
# import os
# HUB_REPO_ID = "your-user/atc-grpo-medium-7b"
# api = HfApi(token=os.environ.get("HF_TOKEN"))
# api.upload_folder(folder_path=str(OUTPUT_DIR), repo_id=HUB_REPO_ID, repo_type="model")
# print("Uploaded to", HUB_REPO_ID)
print("Hub upload cell is commented out by default.")


## Troubleshooting

| Issue | What to do |
|-------|------------|
| CUDA OOM on 7B | Lower `GRPO_EPISODES`; in SFT use `--batch_size 1`; smaller model or larger GPU |
| `trl` / `GRPOConfig` keyword errors | Restart kernel; pin `trl` to match your Unsloth build |
| Jupyter: Unsloth import fails after install | Kernel → Restart once, then re-run from §1 (skip pip if already satisfied) |
| Clone permission denied | Use a **public** `REPO_URL` or set `USE_LOCAL_REPO=True` and work from a local git checkout |
| `eval.py` cannot load trained folder | Ensure `OUTPUT_DIR` contains `adapter_config.json` from GRPO save |
